# Visualize an MITgcm Global Ocean run

Reads the Zarr written by `datagen.mitgcm.global_ocean.scripts.run` or
`datagen.mitgcm.global_ocean.scripts.preflight` and renders snapshots,
Hovmöller diagrams, diagnostics, and animations of the 4° global-ocean
tutorial state.

Dataset layout: `(time, field, lat, lon)` with
`field ∈ {theta_k1, salt_k1, u_k2, v_k2, eta}`. Time is in seconds since
the first saved state in the MITgcm output.

## Environment

Needs `xarray`, `zarr`, `numpy`, `matplotlib`, `cartopy`, `imageio[ffmpeg]`.
All ship with the `datagen` project env.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import os

zarr_override = os.environ.get("GLOBAL_OCEAN_ZARR")
if zarr_override:
    DATA_PATH = Path(zarr_override)
else:
    DATA_ROOT = Path(os.environ.get(
        "DATA_ROOT",
        f"{os.environ.get('SCRATCH', '/tmp')}/fots-data/mitgcm/global-ocean",
    ))
    CORNER = os.environ.get("CORNER", "corner_00")
    DATA_PATH = DATA_ROOT / "preflight" / CORNER / "run.zarr"

ds = xr.open_zarr(DATA_PATH)
ds

In [ ]:
print("Path:", DATA_PATH)
print("Dims:", dict(ds.sizes))
print("Time range:", float(ds.time.min()) / 86400, "..",
      float(ds.time.max()) / 86400, "days")
print("Lat range: ", float(ds.lat.min()), "..", float(ds.lat.max()), "deg")
print("Lon range: ", float(ds.lon.min()), "..", float(ds.lon.max()), "deg")
print("Fields:    ", [str(f) for f in ds.field.values])
print("Parameters:")
for k, v in ds.attrs.items():
    if k.startswith("param_"):
        print(f"  {k[6:]:22s} = {v}")

## Snapshot of the extracted ocean fields

First saved state from the run. The tutorial output includes near-surface
potential temperature and salinity, level-2 horizontal velocity, and sea
surface height.

In [ ]:
lat = ds.lat.values
lon = ds.lon.values

def field(name):
    return ds.fields.sel(field=name)

def plot_field(ax, arr, title, cmap="RdBu_r", symmetric=True, units=""):
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    if not finite.any():
        ax.set_title(title)
        ax.text(0.5, 0.5, "all values are NaN", ha="center", va="center",
                transform=ax.transAxes)
        return None
    if symmetric:
        vmax = float(np.nanmax(np.abs(arr)))
        vmin = -vmax
    else:
        vmin, vmax = float(np.nanmin(arr)), float(np.nanmax(arr))
    im = ax.pcolormesh(lon, lat, arr, cmap=cmap, vmin=vmin, vmax=vmax,
                       shading="auto")
    ax.set_title(title)
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label=units)
    return im

t0 = 0
fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True)
plot_field(axes[0, 0], field("theta_k1").isel(time=t0).values,
           "potential temperature k=1", cmap="viridis", symmetric=False,
           units="degC")
plot_field(axes[0, 1], field("salt_k1").isel(time=t0).values,
           "salinity k=1", cmap="magma", symmetric=False, units="psu")
plot_field(axes[0, 2], field("eta").isel(time=t0).values,
           "sea surface height", cmap="RdBu_r", symmetric=True, units="m")
plot_field(axes[1, 0], field("u_k2").isel(time=t0).values,
           "u velocity k=2", units="m/s")
plot_field(axes[1, 1], field("v_k2").isel(time=t0).values,
           "v velocity k=2", units="m/s")
speed0 = np.sqrt(field("u_k2").isel(time=t0).values ** 2 +
                 field("v_k2").isel(time=t0).values ** 2)
plot_field(axes[1, 2], speed0, "speed k=2", cmap="viridis",
           symmetric=False, units="m/s")
fig.suptitle(f"t = {float(ds.time.isel(time=t0)) / 86400:.2f} days")
plt.show()

## Time evolution of sea surface height

Evenly-spaced panels of `eta`, useful for checking that the preflight run
is producing coherent large-scale ocean adjustment rather than an immediate
numerical failure.

In [ ]:
eta_all = field("eta").values
n_times = ds.sizes["time"]
n_panels = min(6, n_times)
panels = np.linspace(0, n_times - 1, n_panels, dtype=int)
vmax_eta = float(np.nanmax(np.abs(eta_all)))

fig, axes = plt.subplots(2, 3, figsize=(18, 8),
                         constrained_layout=True, sharex=True, sharey=True)
for ax in axes.flat[n_panels:]:
    ax.axis("off")
for ax, ti in zip(axes.flat, panels):
    im = ax.pcolormesh(lon, lat, eta_all[ti], cmap="RdBu_r",
                       vmin=-vmax_eta, vmax=vmax_eta, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=ti)) / 86400:.2f} d")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="eta [m]")
fig.suptitle("Sea surface height")
plt.show()

## Zonal-mean Hovmöller diagrams

Zonal means over longitude as functions of `(time, lat)`. These are compact
checks for broad thermal, haline, and surface-height drift during a corner
preflight.

In [ ]:
theta_zm = field("theta_k1").mean("lon")
salt_zm = field("salt_k1").mean("lon")
eta_zm = field("eta").mean("lon")
t_d = ds.time.values / 86400.0

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
im0 = axes[0].pcolormesh(t_d, lat, theta_zm.T, cmap="viridis",
                         shading="auto")
axes[0].set_title("Zonal-mean theta k=1 [degC]")
axes[0].set_xlabel("time [days]")
axes[0].set_ylabel("lat [deg]")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(t_d, lat, salt_zm.T, cmap="magma", shading="auto")
axes[1].set_title("Zonal-mean salinity k=1 [psu]")
axes[1].set_xlabel("time [days]")
axes[1].set_ylabel("lat [deg]")
fig.colorbar(im1, ax=axes[1])

vmax = float(np.nanmax(np.abs(eta_zm)))
im2 = axes[2].pcolormesh(t_d, lat, eta_zm.T, cmap="RdBu_r",
                         vmin=-vmax, vmax=vmax, shading="auto")
axes[2].set_title("Zonal-mean eta [m]")
axes[2].set_xlabel("time [days]")
axes[2].set_ylabel("lat [deg]")
fig.colorbar(im2, ax=axes[2])
plt.show()

## Globally-integrated diagnostics

Area-weighted means of surface temperature, salinity, and sea surface height,
plus the global maximum level-2 speed.

In [ ]:
lat_rad = np.deg2rad(lat)
w = np.cos(lat_rad)
w = w / w.sum()

theta = field("theta_k1").values
salt = field("salt_k1").values
u = field("u_k2").values
v = field("v_k2").values
eta = field("eta").values
speed = np.sqrt(u ** 2 + v ** 2)

theta_mean = theta.mean(axis=-1) @ w
salt_mean = salt.mean(axis=-1) @ w
eta_mean = eta.mean(axis=-1) @ w
max_speed = speed.max(axis=(-1, -2))

fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)
axes[0].plot(t_d, theta_mean)
axes[0].set_xlabel("time [days]")
axes[0].set_ylabel("<theta> [degC]")
axes[0].set_title("Area-weighted mean theta")
axes[1].plot(t_d, salt_mean)
axes[1].set_xlabel("time [days]")
axes[1].set_ylabel("<salt> [psu]")
axes[1].set_title("Area-weighted mean salinity")
axes[2].plot(t_d, eta_mean)
axes[2].set_xlabel("time [days]")
axes[2].set_ylabel("<eta> [m]")
axes[2].set_title("Area-weighted mean sea level")
axes[3].plot(t_d, max_speed)
axes[3].set_xlabel("time [days]")
axes[3].set_ylabel("max speed [m/s]")
axes[3].set_title("Global max speed k=2")
plt.show()

## Sea-surface-height animation (flat MP4)

In [ ]:
import imageio.v3 as iio
from tqdm import trange

fps = 10
dpi = 100
output_path = Path("mitgcm_global_ocean_eta.mp4")

frames = []
for i in trange(eta_all.shape[0]):
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True, dpi=dpi)
    mesh = ax.pcolormesh(lon, lat, eta_all[i], cmap="RdBu_r",
                         vmin=-vmax_eta, vmax=vmax_eta, shading="auto")
    ax.set_title(f"eta | t = {float(ds.time.isel(time=i)) / 86400:.2f} d")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    fig.colorbar(mesh, ax=ax, label="eta [m]")

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")

## Sea-surface-height animation (spherical, rotating globe)

Cartopy orthographic projection with a slow rotation around the central
longitude over the run.

In [ ]:
import cartopy.crs as ccrs
from tqdm import trange

fps = 10
dpi = 100
output_path = Path("mitgcm_global_ocean_eta_spherical.mp4")

num_frames = eta_all.shape[0]
frames = []

for i in trange(num_frames):
    fig = plt.figure(figsize=(8, 8), constrained_layout=True, dpi=dpi)
    cent_lon = (i / num_frames) * 360.0 - 180.0
    ax = fig.add_subplot(
        1, 1, 1,
        projection=ccrs.Orthographic(central_longitude=cent_lon, central_latitude=20.0),
    )
    ax.set_global()
    ax.gridlines(alpha=0.3, linestyle="--")

    mesh = ax.pcolormesh(
        lon, lat, eta_all[i],
        cmap="RdBu_r",
        vmin=-vmax_eta, vmax=vmax_eta,
        shading="auto",
        transform=ccrs.PlateCarree(),
    )
    ax.set_title(f"MITgcm global ocean eta | t = {float(ds.time.isel(time=i)) / 86400:.2f} d")
    fig.colorbar(mesh, ax=ax, shrink=0.6, label="eta [m]", pad=0.05)

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")